In [22]:

%load_ext autoreload
%autoreload 2

import pandas as pd
from optbinning import OptimalBinning
import numpy as np

train = pd.read_parquet("../data/processed/train.parquet")

var = "EXT_SOURCE_3"
optb = OptimalBinning(name=var, dtype="numerical", monotonic_trend="auto")
optb.fit(train[var], train["TARGET"])

optb.binning_table.build()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.18)",10885,0.050567,8509,2376,0.218282,-1.156807,0.108429,0.012845
1,"[0.18, 0.31)",20486,0.095170,17582,2904,0.141755,-0.631727,0.049442,0.006080
2,"[0.31, 0.37)",11009,0.051144,9893,1116,0.101372,-0.250437,0.003563,0.000444
3,"[0.37, 0.41)",10897,0.050623,9843,1054,0.096724,-0.198345,0.002164,0.000270
4,"[0.41, 0.46)",11364,0.052793,10477,887,0.078054,0.036579,0.000070,0.000009
5,"[0.46, 0.50)",12169,0.056532,11309,860,0.070671,0.143908,0.001102,0.000138
6,"[0.50, 0.57)",20527,0.095360,19318,1209,0.058898,0.33873,0.009501,0.001182
7,"[0.57, 0.64)",22025,0.102320,20957,1068,0.048490,0.544172,0.024187,0.002987
8,"[0.64, 0.70)",21536,0.100048,20671,865,0.040165,0.741244,0.040534,0.004954
9,"[0.70, 0.76)",17746,0.082441,17130,616,0.034712,0.892826,0.045640,0.005523


## Set up max_n_bins = 8

In [23]:
optb = OptimalBinning(
    name=var,
    dtype="numerical",
    monotonic_trend="auto",
    max_n_bins=8,         # at most 8 bins
    min_bin_size=0.05,    # each bin at least 5% of the data
)
optb.fit(train[var], train["TARGET"])

optb.binning_table.build()

,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.18)",10885,0.050567,8509,2376,0.218282,-1.156807,0.108429,0.012845
1,"[0.18, 0.31)",20486,0.095170,17582,2904,0.141755,-0.631727,0.049442,0.006080
2,"[0.31, 0.41)",21906,0.101767,19736,2170,0.099060,-0.224796,0.005651,0.000705
3,"[0.41, 0.50)",23533,0.109325,21786,1747,0.074236,0.090854,0.000869,0.000109
4,"[0.50, 0.57)",20527,0.095360,19318,1209,0.058898,0.33873,0.009501,0.001182
5,"[0.57, 0.64)",22025,0.102320,20957,1068,0.048490,0.544172,0.024187,0.002987
6,"[0.64, 0.70)",21536,0.100048,20671,865,0.040165,0.741244,0.040534,0.004954
7,"[0.70, inf)",31679,0.147168,30629,1050,0.033145,0.940644,0.088760,0.010703
8,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
9,Missing,42680,0.198275,38692,3988,0.093440,-0.16017,0.005440,0.000679


## Screen all variables in one loop

In [24]:
EXCLUDE = ["TARGET", "SK_ID_CURR"]
candidates = [c for c in train.columns if c not in EXCLUDE]

results = []
binners = {}
for var in candidates:
    is_num = pd.api.types.is_numeric_dtype(train[var])
    optb = OptimalBinning(
        name=var,
        dtype="numerical" if is_num else "categorical",
        monotonic_trend="auto" if is_num else None,
        max_n_bins=8,
        min_bin_size=0.05,
    )
    try:
        optb.fit(train[var], train["TARGET"])
        table = optb.binning_table.build()
        results.append({"variable": var, "iv": optb.binning_table.iv,
                        "n_bins": len(table) - 3, "status": optb.status})
        binners[var] = optb
    except Exception as e:
        results.append({"variable": var, "iv": None, "n_bins": None, "status": str(e)})

iv_table = pd.DataFrame(results).sort_values("iv", ascending=False)
iv_table.head(30)

,variable,iv,n_bins,status
38,EXT_SOURCE_3,0.332815,8,OPTIMAL
37,EXT_SOURCE_2,0.319643,8,OPTIMAL
36,EXT_SOURCE_1,0.150084,6,OPTIMAL
14,DAYS_EMPLOYED,0.116156,8,OPTIMAL
7,AMT_GOODS_PRICE,0.092841,8,OPTIMAL
13,DAYS_BIRTH,0.088861,8,OPTIMAL
75,AGE_YEARS,0.088861,8,OPTIMAL
24,OCCUPATION_TYPE,0.078018,5,OPTIMAL
74,CREDIT_GOODS_RATIO,0.073348,6,OPTIMAL
35,ORGANIZATION_TYPE,0.072242,8,OPTIMAL


### Check of expected affordability drivers

CREDIT_INCOME_RATIO, ANNUITY_INCOME_RATIO and AMT_INCOME_TOTAL were expected to be important affordability drivers (see expert long list), but have IV below 0.02. CREDIT_TERM passes the IV threshold (0.027) but got only 2 bins. Their binning tables are inspected below to understand why.

In [25]:
for var in ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM", "AMT_INCOME_TOTAL"]:
    print(var, iv_table.loc[iv_table["variable"] == var, "iv"].values)
    display(binners[var].binning_table.build())

CREDIT_INCOME_RATIO [0.01252876]


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 1.01)",11364,0.052793,10638,726,0.063886,0.252124,0.003021,0.000377
1,"[1.01, 1.53)",19016,0.088341,17585,1431,0.075252,0.076159,0.000496,0.000062
2,"[1.53, 2.00)",23112,0.107369,21296,1816,0.078574,0.02937,0.000091,0.000011
3,"[2.00, 2.27)",11263,0.052324,10323,940,0.083459,-0.036263,0.000070,0.000009
4,"[2.27, 2.50)",11064,0.051399,9994,1070,0.096710,-0.198187,0.002194,0.000274
5,"[2.50, 4.85)",78141,0.363013,71257,6884,0.088097,-0.09542,0.003440,0.000430
6,"[4.85, 8.99)",49955,0.232071,46214,3741,0.074887,0.081417,0.001487,0.000186
7,"[8.99, inf)",11342,0.052691,10573,769,0.067801,0.188455,0.001730,0.000216
8,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
9,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000


ANNUITY_INCOME_RATIO [0.00778653]


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.06)",11822,0.054920,11051,771,0.065217,0.230075,0.002641,0.000329
1,"[0.06, 0.11)",35149,0.163289,32558,2591,0.073715,0.098466,0.001519,0.000190
2,"[0.11, 0.16)",54979,0.255411,50695,4284,0.077921,0.038427,0.000371,0.000046
3,"[0.16, 0.20)",36586,0.169964,33493,3093,0.084541,-0.050318,0.000440,0.000055
4,"[0.20, 0.22)",20366,0.094612,18616,1750,0.085928,-0.068108,0.000452,0.000056
5,"[0.22, 0.27)",23314,0.108308,21236,2078,0.089131,-0.108221,0.001327,0.000166
6,"[0.27, 0.30)",10789,0.050121,9801,988,0.091575,-0.137956,0.001011,0.000126
7,"[0.30, inf)",22244,0.103337,20422,1822,0.081910,-0.015835,0.000026,0.000003
8,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
9,Missing,8,0.000037,8,0,0.000000,0.0,0.000000,0.000000


CREDIT_TERM [0.02714131]


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 10.78)",22797,0.105906,21689,1108,0.048603,0.541735,0.024836,0.003067
1,"[10.78, inf)",192452,0.894057,176183,16269,0.084535,-0.050252,0.002306,0.000288
2,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
3,Missing,8,0.000037,8,0,0.000000,0.0,0.000000,0.000000
Totals,,215257,1.000000,197880,17377,0.080727,,0.027141,0.003355


AMT_INCOME_TOTAL [0.01287805]


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 76374.00)",16648,0.077340,15311,1337,0.080310,0.00563,2.445604e-06,3.057001e-07
1,"[76374.00, 98977.50)",25332,0.117683,23216,2116,0.083531,-0.037199,1.654068e-04,2.067466e-05
2,"[98977.50, 127530.00)",36336,0.168803,33133,3203,0.088149,-0.096071,1.622093e-03,2.026837e-04
3,"[127530.00, 180045.00)",72423,0.336449,66276,6147,0.084876,-0.05465,1.028134e-03,1.285007e-04
4,"[180045.00, 211716.00)",15587,0.072411,14325,1262,0.080965,-0.003205,7.447138e-07,9.308918e-08
5,"[211716.00, 252225.00)",21537,0.100052,19933,1604,0.074476,0.087363,7.361932e-04,9.199490e-05
6,"[252225.00, 297607.50)",11475,0.053308,10704,771,0.067190,0.198171,1.927094e-03,2.404934e-04
7,"[297607.50, inf)",15919,0.073953,14982,937,0.058860,0.339408,7.395943e-03,9.200808e-04
8,Special,0,0.000000,0,0,0.000000,0.0,0.000000e+00,0.000000e+00
9,Missing,0,0.000000,0,0,0.000000,0.0,0.000000e+00,0.000000e+00


**Findings**
- CREDIT_INCOME_RATIO (IV 0.013): non-monotonic, with risk peaking at ratios 2.3–2.5 and lower at both ends. Likely selection bias: high-ratio loans were probably approved only for otherwise strong applicants. Dropped.
- ANNUITY_INCOME_RATIO (IV 0.008): risk rises as expected, from 6.5% for the lowest ratios to 9.2% at ratios 0.27–0.30, but falls back to 8.2% for the highest ratios. The overall effect is small, likely for the same selection reason. Dropped.
- CREDIT_TERM (IV 0.027): two bins only. Short loans (under about 11 payments, 11% of clients) default at 4.9%, versus 8.5% for longer loans. This confirms the expected effect "longer term → higher risk" and works as a simple, explainable split. Kept on the short list.
- AMT_INCOME_TOTAL (IV 0.013): the default rate is nearly flat (8.0–8.8%) up to an income of about 210,000, and only the highest incomes show lower risk (5.9% above about 300,000). Weak overall. Dropped.

The weak results for the affordability variables suggest that Home Credit's approval process already accounted for affordability, so among approved loans these variables add little. Affordability remains important in practice, but its effect cannot be measured well in data containing only approved applicants.

In [26]:
SELECTION_EXCLUDED = {
    "DAYS_BIRTH": "Duplicate of AGE_YEARS",
    "REGION_RATING_CLIENT": "Near-duplicate of REGION_RATING_CLIENT_W_CITY (lower IV)",
    "DAYS_EMPLOYED_ANOM": "Captured by the Missing bin of DAYS_EMPLOYED",
    "AMT_CREDIT": "Correlated with AMT_GOODS_PRICE (lower IV)",
}

shortlist = iv_table[
    (iv_table["iv"] >= 0.02) & (~iv_table["variable"].isin(SELECTION_EXCLUDED))
]

In [27]:
print(shortlist)

                       variable        iv  n_bins   status
38                 EXT_SOURCE_3  0.332815       8  OPTIMAL
37                 EXT_SOURCE_2  0.319643       8  OPTIMAL
36                 EXT_SOURCE_1  0.150084       6  OPTIMAL
14                DAYS_EMPLOYED  0.116156       8  OPTIMAL
7               AMT_GOODS_PRICE  0.092841       8  OPTIMAL
75                    AGE_YEARS  0.088861       8  OPTIMAL
24              OCCUPATION_TYPE  0.078018       5  OPTIMAL
74           CREDIT_GOODS_RATIO  0.073348       6  OPTIMAL
35            ORGANIZATION_TYPE  0.072242       8  OPTIMAL
8              NAME_INCOME_TYPE  0.054792       4  OPTIMAL
27  REGION_RATING_CLIENT_W_CITY  0.052011       3  OPTIMAL
9           NAME_EDUCATION_TYPE  0.048175       2  OPTIMAL
43       DAYS_LAST_PHONE_CHANGE  0.046210       8  OPTIMAL
16              DAYS_ID_PUBLISH  0.038663       8  OPTIMAL
12   REGION_POPULATION_RELATIVE  0.035504       7  OPTIMAL
19               FLAG_EMP_PHONE  0.032725       2  OPTIM

In [28]:
woe_train = pd.DataFrame({
    var: binners[var].transform(train[var], metric="woe")
    for var in shortlist["variable"]
})

corr = woe_train.corr().abs()
pairs = (
    corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    .stack()
    .sort_values(ascending=False)
)
pairs[pairs > 0.6]

NAME_INCOME_TYPE             FLAG_EMP_PHONE                0.784298
REGION_RATING_CLIENT_W_CITY  REGION_POPULATION_RELATIVE    0.704139
ORGANIZATION_TYPE            FLAG_EMP_PHONE                0.674237
                             NAME_INCOME_TYPE              0.636731
AGE_YEARS                    FLAG_EMP_PHONE                0.628143
dtype: float64

### Exclusions after the correlation check

Pairs of WoE-transformed variables with absolute correlation above 0.6 were reviewed. For each pair, the variable with the lower IV, or the weaker business meaning, is excluded:

- **FLAG_EMP_PHONE** appears in three pairs: with NAME_INCOME_TYPE (0.78), ORGANIZATION_TYPE (0.67) and AGE_YEARS (0.63). It mainly captures whether the client is employed or a pensioner, which other variables already cover, and has the lowest IV of the group.
- **REGION_POPULATION_RELATIVE** is correlated with REGION_RATING_CLIENT_W_CITY (0.70), which has the higher IV. Excluding it also addresses the fairness concern noted in the expert long list.
- **NAME_INCOME_TYPE** is correlated with ORGANIZATION_TYPE (0.64). ORGANIZATION_TYPE is kept for its higher IV (0.072 vs 0.055), subject to its bins making business sense.

In [29]:
SELECTION_EXCLUDED.update({
    "FLAG_EMP_PHONE": "Proxy for employed/pensioner; correlated with income type, organisation type and age (lowest IV)",
    "REGION_POPULATION_RELATIVE": "Correlated with REGION_RATING_CLIENT_W_CITY (0.70, lower IV); also a fairness concern",
    "NAME_INCOME_TYPE": "Correlated with ORGANIZATION_TYPE (0.64, lower IV)",
})

In [30]:
shortlist = iv_table[
    (iv_table["iv"] >= 0.02) & (~iv_table["variable"].isin(SELECTION_EXCLUDED))
]
woe_train = woe_train[shortlist["variable"]]

corr = woe_train.corr().abs()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print(len(shortlist), "variables")
print(pairs[pairs > 0.6])
print(shortlist)

21 variables
Series([], dtype: float64)
                       variable        iv  n_bins   status
38                 EXT_SOURCE_3  0.332815       8  OPTIMAL
37                 EXT_SOURCE_2  0.319643       8  OPTIMAL
36                 EXT_SOURCE_1  0.150084       6  OPTIMAL
14                DAYS_EMPLOYED  0.116156       8  OPTIMAL
7               AMT_GOODS_PRICE  0.092841       8  OPTIMAL
75                    AGE_YEARS  0.088861       8  OPTIMAL
24              OCCUPATION_TYPE  0.078018       5  OPTIMAL
74           CREDIT_GOODS_RATIO  0.073348       6  OPTIMAL
35            ORGANIZATION_TYPE  0.072242       8  OPTIMAL
27  REGION_RATING_CLIENT_W_CITY  0.052011       3  OPTIMAL
9           NAME_EDUCATION_TYPE  0.048175       2  OPTIMAL
43       DAYS_LAST_PHONE_CHANGE  0.046210       8  OPTIMAL
16              DAYS_ID_PUBLISH  0.038663       8  OPTIMAL
33       REG_CITY_NOT_WORK_CITY  0.032567       2  OPTIMAL
6                   AMT_ANNUITY  0.031555       8  OPTIMAL
15            DA

## Expected vs observed

| Variable | Expected | IV | Observed | Decision |
|---|---|---|---|---|
| EXT_SOURCE_3 | Higher → lower risk | 0.333 | Confirmed, monotonic | Keep |
| DAYS_EMPLOYED | Longer → lower risk | 0.116 | Confirmed, stronger after cleaning | Keep |
| CREDIT_INCOME_RATIO | Higher → higher risk | 0.013 | Weak, non-monotonic (peak at 2.3–2.5) | Drop |
| ... | | | | |

**Note on affordability ratios:** CREDIT_INCOME_RATIO was expected to be a key affordability driver but showed weak, non-monotonic predictive power. A likely reason is selection bias: the data contains only approved loans, and high-ratio loans were probably approved only for otherwise strong applicants.

**Choice of max_n_bins:** limiting EXT_SOURCE_3 to 8 bins reduced IV only from 0.3335 to 0.3328, while merging neighbouring bins with similar default rates. Fewer bins are simpler and more stable, so max_n_bins = 8 is used for all variables.

In [31]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

final_vars = shortlist["variable"].tolist()
joblib.dump({v: binners[v] for v in final_vars}, MODEL_DIR / "binners.joblib")
iv_table.to_csv(MODEL_DIR / "iv_table.csv", index=False)

### Business review of bins

Binning tables of categorical and doubtful variables, checked for sensible groupings and risk patterns that match the expected effects.

In [32]:
review_vars = [
    "ORGANIZATION_TYPE", "OCCUPATION_TYPE", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "FLAG_DOCUMENT_3", "OWN_CAR_AGE",
]
for var in review_vars:
    print(var)
    pd.set_option("display.max_colwidth", None)
    display(binners[var].binning_table.build())

ORGANIZATION_TYPE


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"[Trade: type 4, Trade: type 5, Trade: type 6, Industry: type 12, Military, Religion, Industry: type 6, Security Ministries, Police, University, Insurance, Bank, XNA, Transport: type 1, School, Culture, Services]",55388,0.257311,52386,3002,0.054199,0.426847,0.039261,0.004871
1,"[Trade: type 2, Industry: type 10, Industry: type 9, Government]",11133,0.051720,10373,760,0.068266,0.18113,0.001573,0.000196
2,"[Medicine, Hotel, Emergency, Kindergarten]",13889,0.064523,12915,974,0.070127,0.15222,0.001403,0.000175
3,"[Transport: type 2, Electricity, Industry: type 7, Other, Industry: type 5, Business Entity Type 1]",19375,0.090009,17918,1457,0.075200,0.076913,0.000516,0.000064
4,"[Mobile, Housing, Postal, Telecom, Business Entity Type 2, Industry: type 11, Industry: type 4]",14077,0.065396,12883,1194,0.084819,-0.053914,0.000194,0.000024
5,"[Industry: type 2, Transport: type 4, Business Entity Type 3, Advertising, Trade: type 7, Legal Services, Trade: type 1]",57851,0.268753,52511,5340,0.092306,-0.146716,0.006153,0.000768
6,"[Trade: type 3, Self-employed, Industry: type 3]",31398,0.145863,28182,3216,0.102427,-0.261968,0.011174,0.001393
7,"[Security, Agriculture, Realtor, Restaurant, Industry: type 8, Industry: type 1, Construction, Cleaning, Industry: type 13, Transport: type 3]",12146,0.056426,10712,1434,0.118064,-0.421616,0.011969,0.001485
8,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
9,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000


OCCUPATION_TYPE


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"[Accountants, IT staff, Managers]",22188,0.103077,20932,1256,0.056607,0.380834,0.012759,0.001585
1,"[High skill tech staff, Private service staff, HR staff, Core staff, Secretaries, Medicine staff, Realty agents]",37035,0.172050,34603,2432,0.065668,0.222713,0.007776,0.000970
2,"[Sales staff, Cleaning staff]",25595,0.118904,23149,2446,0.095566,-0.185016,0.004399,0.000549
3,"[Cooking staff, Laborers, Security staff]",47467,0.220513,42441,5026,0.105884,-0.299023,0.022353,0.002784
4,"[Drivers, Waiters/barmen staff, Low-skill Laborers]",15492,0.071970,13663,1829,0.118061,-0.421591,0.015265,0.001894
5,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
6,Missing,67480,0.313486,63092,4388,0.065027,0.233207,0.015467,0.001929
Totals,,215257,1.000000,197880,17377,0.080727,,0.078018,0.009711


NAME_EDUCATION_TYPE


...

,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"[Unknown, Widow]",11256,0.052291,10590,666,0.059168,0.333862,0.005072,6.310251e-04
1,[Married],137457,0.638572,127100,10357,0.075347,0.074798,0.003462,4.327076e-04
2,[Separated],13854,0.064360,12744,1110,0.080121,0.008187,0.000004,5.374197e-07
3,[Civil marriage],20844,0.096833,18772,2072,0.099405,-0.228661,0.005573,6.951169e-04
4,[Single / not married],31846,0.147944,28674,3172,0.099604,-0.230885,0.008689,1.083738e-03
5,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000e+00
6,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000e+00
Totals,,215257,1.000000,197880,17377,0.080727,,0.022801,2.843125e-03


FLAG_DOCUMENT_3


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.50)",62412,0.289942,58561,3851,0.061703,0.289223,0.021497,0.002678
1,"[0.50, inf)",152845,0.710058,139319,13526,0.088495,-0.100361,0.007460,0.000932
2,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
3,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
Totals,,215257,1.000000,197880,17377,0.080727,,0.028957,0.003610


OWN_CAR_AGE


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 3.50)",13644,0.063385,12827,817,0.059880,0.321155,0.005718,0.000712
1,"[3.50, 8.50)",20280,0.094213,19134,1146,0.056509,0.382676,0.011766,0.001462
2,"[8.50, 12.50)",12779,0.059366,11829,950,0.074341,0.089334,0.000456,0.000057
3,"[12.50, 16.50)",11249,0.052258,10283,966,0.085874,-0.06743,0.000244,0.000031
4,"[16.50, inf)",15214,0.070678,13786,1428,0.093861,-0.165135,0.002066,0.000258
5,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
6,Missing,142091,0.660099,130021,12070,0.084946,-0.05554,0.002084,0.000260
Totals,,215257,1.000000,197880,17377,0.080727,,0.022335,0.002780


In [33]:
SELECTION_EXCLUDED.update({
    "FLAG_DOCUMENT_3": "Undocumented meaning and counterintuitive pattern; cannot be justified as a risk driver",
})

### Business review results

- ORGANIZATION_TYPE, OCCUPATION_TYPE, NAME_EDUCATION_TYPE and OWN_CAR_AGE: groupings and risk patterns make business sense. Kept.
- NAME_FAMILY_STATUS: plausible pattern but weak (IV 0.023) and partly overlapping with age through the "Widow" group. Kept provisionally; to be confirmed in the model. Sensitive characteristic.
- FLAG_DOCUMENT_3: undocumented meaning and counterintuitive pattern. Dropped.
- Some ORGANIZATION_TYPE categories are anonymised (e.g. "Trade: type 4"), so their grouping is justified by data only.

In [34]:
check = ["NAME_CONTRACT_TYPE", "CNT_CHILDREN", "NAME_HOUSING_TYPE",
         "FLAG_OWN_REALTY", "FLAG_OWN_CAR", "AMT_REQ_CREDIT_BUREAU_YEAR"]
print(iv_table[iv_table["variable"].isin(check)].round(3))

                      variable     iv  n_bins   status
69  AMT_REQ_CREDIT_BUREAU_YEAR  0.019       5  OPTIMAL
11           NAME_HOUSING_TYPE  0.015       2  OPTIMAL
0           NAME_CONTRACT_TYPE  0.014       2  OPTIMAL
1                 FLAG_OWN_CAR  0.006       2  OPTIMAL
3                 CNT_CHILDREN  0.006       3  OPTIMAL
2              FLAG_OWN_REALTY  0.000       2  OPTIMAL


## Expected vs observed

| Variable | Expected | IV | Observed | Decision |
|---|---|---|---|---|
| EXT_SOURCE_3 | Higher → lower risk | 0.333 | Confirmed, monotonic | Keep |
| EXT_SOURCE_2 | Higher → lower risk | 0.320 | Confirmed, monotonic | Keep |
| EXT_SOURCE_1 | Higher → lower risk | 0.150 | Confirmed | Keep |
| DAYS_EMPLOYED | Longer → lower risk | 0.116 | Confirmed, stronger after cleaning | Keep |
| AGE_YEARS | Older → lower risk | 0.089 | Confirmed | Keep |
| OCCUPATION_TYPE | Varies by occupation | 0.078 | Managers/IT 5.7% vs low-skill 11.8% | Keep |
| CREDIT_GOODS_RATIO | Higher → higher risk | 0.073 | Confirmed | Keep |
| ORGANIZATION_TYPE | Varies by industry | 0.072 | 58 categories grouped into 8 sensible risk groups | Keep |
| REGION_RATING_CLIENT | Worse rating → higher risk | 0.052 | Confirmed (W_CITY version kept) | Keep |
| NAME_EDUCATION_TYPE | More education → lower risk | 0.048 | Confirmed: higher education 5.4% vs 8.9% | Keep |
| AMT_ANNUITY | Higher payment → higher risk | 0.032 | Weak | Keep |
| CREDIT_TERM | Longer → higher risk | 0.027 | Confirmed: short loans 4.9% vs 8.5% | Keep |
| NAME_FAMILY_STATUS | Married → lower risk | 0.023 | Confirmed but weak; partly overlaps with age | Keep provisionally |
| DAYS_LAST_PHONE_CHANGE | Recent → higher risk | 0.046 | Added after EDA; confirmed | Keep |
| DAYS_ID_PUBLISH | Recent → higher risk | 0.039 | Added after EDA; confirmed | Keep |
| REG_CITY_NOT_WORK_CITY | Mismatch → higher risk | 0.033 | Added after EDA; confirmed | Keep |
| REG_CITY_NOT_LIVE_CITY | Mismatch → higher risk | 0.023 | Added after EDA; confirmed | Keep |
| AMT_CREDIT | Larger → higher risk | 0.058 | Correlated with AMT_GOODS_PRICE | Drop |
| NAME_INCOME_TYPE | Pensioners lower risk | 0.055 | Correlated with ORGANIZATION_TYPE | Drop |
| REGION_POPULATION_RELATIVE | Uncertain | 0.036 | Correlated with regional rating; fairness concern | Drop |
| AMT_REQ_CREDIT_BUREAU_YEAR | More → higher risk | 0.019 | Just below threshold | Drop |
| NAME_HOUSING_TYPE | Own home → lower risk | 0.015 | Weak | Drop |
| NAME_CONTRACT_TYPE | Cash loans higher risk | 0.014 | Weak: difference explained by loan size | Drop |
| CREDIT_INCOME_RATIO | Higher → higher risk | 0.013 | Weak, non-monotonic | Drop |
| AMT_INCOME_TOTAL | Higher → lower risk | 0.013 | Only highest incomes differ | Drop |
| ANNUITY_INCOME_RATIO | Higher → higher risk | 0.008 | Weak | Drop |
| CNT_CHILDREN | More → higher risk | 0.006 | Weak | Drop |
| FLAG_OWN_CAR | Owner → lower risk | 0.006 | Weak; car age is more informative | Drop |
| FLAG_OWN_REALTY | Owner → lower risk | 0.000 | No predictive power | Drop |

**Variables kept that were not on the long list:** AMT_GOODS_PRICE (0.093; the long list planned it only as a ratio, but it is the strongest loan-size variable), DAYS_REGISTRATION (0.030) and OWN_CAR_AGE (0.022) passed IV screening and the business review (keep)

**Summary:** most long-list expectations were confirmed. The main surprise is the weakness of the affordability variables, likely due to selection bias in approved-only data. Asset ownership was also expected to lower risk, but FLAG_OWN_REALTY shows no predictive power (IV 0.000) and FLAG_OWN_CAR is weak (0.006). Only the age of the car carries some information.

### Variables not on the expert long list

These passed IV screening without a prior expectation, or were created during cleaning. They were reviewed for business sense before a decision was made.

| Variable | IV | Observed | Decision |
|---|---|---|---|
| AMT_GOODS_PRICE | 0.093 | Strongest loan-size variable; the long list planned it only as a ratio | Keep |
| DAYS_REGISTRATION | 0.030 | Passed screening and business review | Keep |
| OWN_CAR_AGE | 0.022 | Newer cars 5.7–6.0% vs old cars 9.4%; non-owners near average | Keep |
| FLAG_DOCUMENT_3 | 0.029 | Undocumented meaning; clients who provided it default more (8.8% vs 6.2%) | Drop |
| FLAG_EMP_PHONE | 0.033 | Proxy for employed/pensioner; correlated with three kept variables | Drop |
| DAYS_EMPLOYED_ANOM | 0.033 | Captured by the Missing bin of DAYS_EMPLOYED | Drop |
| DAYS_BIRTH | 0.089 | Duplicate of AGE_YEARS | Drop |

## Totally 20 variables keep

In [35]:
shortlist = iv_table[
    (iv_table["iv"] >= 0.02) & (~iv_table["variable"].isin(SELECTION_EXCLUDED))
]
print(len(shortlist))

20


## Save binners

In [36]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

final_vars = shortlist["variable"].tolist()
joblib.dump({v: binners[v] for v in final_vars}, MODEL_DIR / "binners.joblib")
iv_table.to_csv(MODEL_DIR / "iv_table.csv", index=False)
pd.Series(SELECTION_EXCLUDED, name="reason").to_csv(MODEL_DIR / "selection_excluded.csv")

print(f"Saved {len(final_vars)} binners")

Saved 20 binners
